# 배터리 정책 분석 (점유 정책 사용) — 균등성 / 표 / 병렬 GIF

`colab_runner_occupancy`가 학습한 **점유(occupancy) 정책**을 그대로 사용한다.

- **NB**: `{EXP_TAG}_occ_nobatt_base` (배터리 없음 + 점유)
- **B** : `{EXP_TAG}_occ_batt_base`  (배터리 + 점유)

**배터리 회계 방식**: 두 정책은 관측 차원이 달라(85 vs 99) 같은 환경에서 직접 섞을 수 없다.
대신 **각 정책을 자기 환경에서 굴리고, 매 스텝 드론별 행동(이동/정지)으로부터 배터리를
오프라인으로 동일하게 시뮬레이션**한다 (이동 = move_cost, 정지 = hover_cost). 정책 행동만
다르고 배터리 회계는 완전히 동일 → 공정 비교.

## 1. clone / drive / pip

In [ ]:
BRANCH = "Saehoon"
%cd /content
!rm -rf /content/RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git /content/RL-2026s1-tp
%cd /content/RL-2026s1-tp
!git switch $BRANCH && git pull origin $BRANCH
!git log --oneline -1
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery_letters"
!mkdir -p {DRIVE_ROOT}/analysis {DRIVE_ROOT}/gifs
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow

## 2. 설정 — 점유 ckpt 경로

`colab_runner_occupancy`에서 학습한 케이스 폴더를 가리킨다. 배터리 회계용 비용은
batt 정책이 학습된 값(여기서는 relaxed)과 동일하게 둔다.

In [ ]:
TARGET_SEQUENCE = "GROUND,D,G"
EXP_TAG = "DG"
GRID_SIZE, N_AGENTS, MAX_STEPS = 25, 14, 500
SHAPES = [s.strip() for s in TARGET_SEQUENCE.split(",") if s.strip()]
COMPLETION_REWARD = 120.0

NB_DIR = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_occ_nobatt_base"   # 배터리 없음 + 점유
B_DIR  = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_occ_batt_base"     # 배터리 + 점유

# 배터리 오프라인 회계 비용 (batt 정책 학습값과 일치시키기)
SIM = dict(init=1.0, hover=0.001, move=0.0025)
# batt 정책 환경 생성 시 쓰는 배터리 파라미터(관측에 들어가는 잔량이 학습과 같아야 함)
BATT_ENV = dict(init=1.0, hover=0.001, move=0.0025, pen=0.05)

## 3. 헬퍼 — 정책별 환경/롤아웃 + 오프라인 배터리

각 정책을 자기(점유) 환경에서 굴리며 매 스텝 행동을 읽어 배터리를 동일 규칙으로 시뮬레이션.

In [ ]:
import os, re, glob, torch, numpy as np
from torchrl.envs.utils import ExplorationType, set_exploration_type, step_mdp
import comm_eval as ce
import comm_eval_battery as ceb
from comm_env_occupancy import OccupancyShapeFormationEnv, BatteryOccupancyShapeFormationEnv
GROUP = ceb.GROUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def latest_ckpt(d):
    cand=[]
    for p in glob.glob(os.path.join(d,"ckpt_*.pt")):
        m=re.search(r"ckpt_(\d+)\.pt$",p)
        if m: cand.append((int(m.group(1)),p))
    if not cand: return None,0
    cand.sort(); return cand[-1][1],cand[-1][0]
def eval_ckpt(d):
    b=os.path.join(d,"ckpt_best.pt")
    return b if os.path.exists(b) else latest_ckpt(d)[0]

def make_env_for(kind, seed=0):
    if kind == "nobatt":
        ce.ShapeFormationEnv = OccupancyShapeFormationEnv          # 점유(배터리X) 환경
        return ce.make_env(seed=seed, device=device, grid_size=GRID_SIZE, n_agents=N_AGENTS,
                           max_steps=MAX_STEPS, shapes=SHAPES, comm_fail_prob=0.0,
                           completion_reward=COMPLETION_REWARD)
    else:
        ceb.BatteryShapeFormationEnv = BatteryOccupancyShapeFormationEnv  # 점유+배터리 환경
        return ceb.make_env(seed=seed, device=device, grid_size=GRID_SIZE, n_agents=N_AGENTS,
                            max_steps=MAX_STEPS, shapes=SHAPES, comm_fail_prob=0.0,
                            completion_reward=COMPLETION_REWARD, wind_prob=0.0, wind_strength=1,
                            randomize_wind=False, initial_battery=BATT_ENV["init"],
                            hover_battery_cost=BATT_ENV["hover"], move_battery_cost=BATT_ENV["move"],
                            low_battery_move_penalty=BATT_ENV["pen"])

def load_actor(kind, ckpt):
    env, base = make_env_for(kind, seed=0)
    a = ceb.build_actor(base.obs_dim, 5, N_AGENTS, 128, device)
    with torch.no_grad(): a(env.reset())
    a.load_state_dict(torch.load(ckpt, map_location=device)["actor"]); a.eval()
    return a

def run_episode(kind, actor, greedy=False, record=False, seed=0):
    env, base = make_env_for(kind, seed=seed)
    td = env.reset()
    expl = ExplorationType.MODE if greedy else ExplorationType.RANDOM
    batt = {ag: SIM["init"] for ag in base.possible_agents}   # 오프라인 배터리(두 정책 동일 회계)
    frames = []
    def snap(): return dict(positions=dict(base.agent_pos), batteries=dict(batt),
                            target_cells=list(base.target_cells))
    if record: frames.append(snap())
    success = False
    for _ in range(base.max_steps):
        with set_exploration_type(expl), torch.no_grad(): actor(td)
        acts = td.get((GROUP, "action")).reshape(-1).cpu().numpy()
        td = env.step(td)
        for i, ag in enumerate(base.possible_agents):
            batt[ag] = max(0.0, batt[ag] - (SIM["move"] if int(acts[i]) != 0 else SIM["hover"]))
        if record: frames.append(snap())
        if bool(td.get(("next", GROUP, "done")).all().item()):
            success = bool(td.get(("next", GROUP, "terminated")).any().item()); break
        td = step_mdp(td)
    cov = getattr(base, "last_occupied_count", 0) / max(1, len(base.target_cells))
    return dict(success=success, coverage=cov, final_batt=dict(batt), frames=frames)

nb_actor = load_actor("nobatt", eval_ckpt(NB_DIR))
b_actor  = load_actor("batt",   eval_ckpt(B_DIR))
print("로드 완료 — NB:", eval_ckpt(NB_DIR), "| B:", eval_ckpt(B_DIR))

## 4. 성능 표 (NB vs 배터리 정책)

성공률(greedy/stochastic)·평균 커버리지·최종(시뮬) 배터리 평균.

In [ ]:
N_EVAL = 50
def perf(kind, actor):
    def rate(g):
        rs=[run_episode(kind, actor, greedy=g, seed=1000+i) for i in range(N_EVAL)]
        return (np.mean([r["success"] for r in rs]), np.mean([r["coverage"] for r in rs]),
                np.mean([np.mean(list(r["final_batt"].values())) for r in rs]))
    sg,cg,_ = rate(True); ss,cs,fb = rate(False)
    return sg, ss, (cg+cs)/2, fb
out=[]
def emit(x): out.append(x); print(x)
emit(f"=== 성능 비교 (n={N_EVAL}, {TARGET_SEQUENCE}, 점유 정책) ===")
hdr="정책".ljust(16)+"succ(greedy)".rjust(13)+"succ(stoch)".rjust(13)+"coverage".rjust(11)+"final_batt(sim)".rjust(16)
emit(hdr); emit("-"*len(hdr))
for name,kind,actor in [("NB(배터리없음)","nobatt",nb_actor), ("B(배터리)","batt",b_actor)]:
    sg,ss,cov,fb=perf(kind,actor)
    emit(name.ljust(16)+f"{sg*100:.0f}%".rjust(13)+f"{ss*100:.0f}%".rjust(13)+f"{cov*100:.1f}%".rjust(11)+f"{fb*100:.1f}%".rjust(16))
open(f"{DRIVE_ROOT}/analysis/perf_{EXP_TAG}.txt","w",encoding="utf-8").write("\n".join(out)+"\n")
print("saved ->", f"{DRIVE_ROOT}/analysis/perf_{EXP_TAG}.txt")

## 5. 균등성 — 드론별 최종 배터리 분포 + box plot

두 정책을 N판씩 돌려(동일 오프라인 회계) 드론별 최종 배터리를 모은다.
균등할수록 표준편차↓·최소↑·Jain↑.

In [ ]:
import matplotlib.pyplot as plt
N_FAIR = 40
def collect(kind, actor):
    return np.array([np.array(list(run_episode(kind, actor, greedy=False, seed=2000+i)["final_batt"].values()))
                     for i in range(N_FAIR)])
NBv, Bv = collect("nobatt", nb_actor), collect("batt", b_actor)
def jain(x):
    s1=x.sum(1); s2=(x**2).sum(1); return float(np.mean(s1**2/(x.shape[1]*np.maximum(s2,1e-9))))
def stats(v): return dict(mean=v.mean(), std=float(np.mean(v.std(1))), mn=float(np.mean(v.min(1))),
                          spread=float(np.mean(v.max(1)-v.min(1))), jain=jain(v))
sN,sB=stats(NBv),stats(Bv)
print(f"{'지표':<14}{'NB':>12}{'배터리정책':>14}")
for k,lab in [('mean','평균잔량'),('std','표준편차↓'),('mn','최소잔량↑'),('spread','폭(max-min)↓'),('jain','Jain↑')]:
    f=(lambda x: f"{x:.3f}") if k=='jain' else (lambda x: f"{x*100:.1f}%")
    print(f"{lab:<14}{f(sN[k]):>12}{f(sB[k]):>14}")
plt.figure(figsize=(6,5))
plt.boxplot([NBv.flatten()*100, Bv.flatten()*100], labels=["NB policy","battery policy"], showmeans=True)
plt.ylabel("final battery (%)"); plt.title(f"Per-drone final battery  [{TARGET_SEQUENCE}]"); plt.grid(axis="y",alpha=0.3)
png=f"{DRIVE_ROOT}/analysis/fairness_box_{EXP_TAG}.png"; plt.tight_layout(); plt.savefig(png,dpi=120); plt.show()
print("saved ->", png)

## 6. 병렬 GIF — 잔량 급감 드론 비교

NB에서 최종 잔량이 가장 낮은(가장 많이 소모) 드론을 골라, NB와 배터리 정책을 나란히 표시.
해당 드론에 자홍색 링 + ★ 표식 → NB는 계속 활발히 움직여 잔량 급감, 배터리 정책은 절약.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from matplotlib.animation import FuncAnimation, PillowWriter
from comm_eval_battery import _battery_color as bcolor

SEED=7
nb_run=run_episode("nobatt", nb_actor, greedy=False, record=True, seed=SEED)
b_run =run_episode("batt",   b_actor,  greedy=False, record=True, seed=SEED)
mark=min(nb_run["final_batt"], key=nb_run["final_batt"].get)
print("표식 드론:", mark, "| NB 최종:", f"{nb_run['final_batt'][mark]*100:.0f}%",
      "| B 최종:", f"{b_run['final_batt'][mark]*100:.0f}%")

def save_parallel(hA,hB,tA,tB,mark,path,fps=4):
    n=max(len(hA),len(hB)); fig,ax=plt.subplots(1,2,figsize=(13,7),facecolor="#0a0a14")
    def one(a,h,step,title):
        f=h[min(step,len(h)-1)]; a.clear(); a.set_facecolor("#0a0a14")
        a.set_xlim(-0.5,GRID_SIZE-0.5); a.set_ylim(-0.5,GRID_SIZE-0.5); a.invert_yaxis()
        a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
        ts=set(f["target_cells"]); mb=f["batteries"].get(mark,0.0)
        a.set_title(f"{title}\nstep {min(step,len(h)-1)} | {mark} batt={mb*100:.0f}%", color="#ddd", fontsize=11)
        for (r,c) in f["target_cells"]:
            a.add_patch(patches.Rectangle((c-0.5,r-0.5),1,1,facecolor="#1c1c2e",edgecolor="#3a3a55",lw=0.6,ls=(0,(2,2))))
        for ag,(r,c) in f["positions"].items():
            on=(r,c) in ts
            a.add_patch(patches.Circle((c,r),0.34 if on else 0.22,facecolor="#ffd24a" if on else "#3a3a44",
                                       edgecolor="#fff4b3" if on else "#555",lw=1.2))
            b=f["batteries"].get(ag,0.0)
            a.add_patch(patches.Rectangle((c-0.45,r-0.78),0.9,0.13,facecolor="#222230",edgecolor="#777",lw=0.3))
            if b>0: a.add_patch(patches.Rectangle((c-0.45,r-0.78),0.9*max(0,min(1,b)),0.13,facecolor=bcolor(b),edgecolor="none"))
            if ag==mark:
                a.add_patch(patches.Circle((c,r),0.62,facecolor="none",edgecolor="#ff45ff",lw=2.6))
                a.text(c,r+0.85,"★",color="#ff45ff",ha="center",va="top",fontsize=12,fontweight="bold")
    def draw(s): one(ax[0],hA,s,tA); one(ax[1],hB,s,tB)
    anim=FuncAnimation(fig,draw,frames=n,interval=1000//fps); anim.save(path,writer=PillowWriter(fps=fps)); plt.close(fig)

gif=f"{DRIVE_ROOT}/gifs/parallel_{EXP_TAG}_{mark}.gif"
save_parallel(nb_run["frames"], b_run["frames"], "NB policy (배터리 무시)", "battery policy (절약)", mark, gif)
from IPython.display import Image, display
print("saved ->", gif); display(Image(gif))